# IT3051 - Fundamentals of Data Mining & Machine Learning
## Mini Project 2026: Used Car Price Prediction System
### Stage 6: Model Development & Stage 7: Optimization — Gradient Boosting Regressor

**Assigned Model:** Gradient Boosting Regressor (Sequential Boosting Ensemble)  
**Target Variable:** `log_price` (log-transformed vehicle selling price)  
**Input Features:** 13 engineered and scaled features from `Used_Car_Price_Prediction_System.ipynb`

In [8]:
# 1. Import Core Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

pd.set_option('display.float_format', '{:,.4f}'.format)

---
## 2. Load Preprocessed Datasets

In [9]:
# Load preprocessed datasets
X_train = pd.read_csv('X_train.csv')
X_test  = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').values.ravel()
y_test  = pd.read_csv('y_test.csv').values.ravel()

print(f"X_train shape: {X_train.shape} (3,206 samples, 13 features)")
print(f"X_test  shape: {X_test.shape} (802 samples, 13 features)")

X_train shape: (3206, 13) (3,206 samples, 13 features)
X_test  shape: (802, 13) (802 samples, 13 features)


---
## 3. Baseline Gradient Boosting & 5-Fold Cross-Validation
We train a baseline Gradient Boosting Regressor (`n_estimators=100`, `learning_rate=0.1`, `max_depth=3`, `random_state=42`) using **5-Fold Cross-Validation** on training data only.

In [10]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
base_gbr = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

base_cv = cross_validate(
    base_gbr,
    X_train,
    y_train,
    cv=kf,
    scoring={
        'r2': 'r2',
        'mae': 'neg_mean_absolute_error',
        'rmse': 'neg_root_mean_squared_error'
    },
    n_jobs=-1
)

base_cv_r2   = base_cv['test_r2'].mean()
base_cv_r2_s = base_cv['test_r2'].std()
base_cv_mae  = -base_cv['test_mae'].mean()
base_cv_rmse = -base_cv['test_rmse'].mean()

# Fit and test baseline on untouched test set
base_gbr.fit(X_train, y_train)
y_pred_base = base_gbr.predict(X_test)

base_test_r2   = r2_score(y_test, y_pred_base)
base_test_mae  = mean_absolute_error(y_test, y_pred_base)
base_test_rmse = root_mean_squared_error(y_test, y_pred_base)

y_test_dollar = np.expm1(y_test)
y_pred_base_dollar = np.expm1(y_pred_base)
base_test_mae_dollar  = mean_absolute_error(y_test_dollar, y_pred_base_dollar)
base_test_rmse_dollar = root_mean_squared_error(y_test_dollar, y_pred_base_dollar)

print("=== Baseline Gradient Boosting Performance ===")
print(f"CV R²                 : {base_cv_r2:.4f} (+/- {base_cv_r2_s:.4f})")
print(f"CV MAE                : {base_cv_mae:.4f}")
print(f"CV RMSE               : {base_cv_rmse:.4f}")
print(f"Test R²               : {base_test_r2:.4f}")
print(f"Test MAE ($ scale)    : ${base_test_mae_dollar:,.2f}")
print(f"Test RMSE ($ scale)   : ${base_test_rmse_dollar:,.2f}")

=== Baseline Gradient Boosting Performance ===
CV R²                 : 0.6650 (+/- 0.0211)
CV MAE                : 0.3752
CV RMSE               : 0.4881
Test R²               : 0.6619
Test MAE ($ scale)    : $18,291.22
Test RMSE ($ scale)   : $39,318.53


---
## 4. Feature Importance Analysis

In [11]:
# Feature importances
importances = pd.Series(base_gbr.feature_importances_, index=X_train.columns).sort_values(ascending=False)

os.makedirs(os.path.join('results', 'feature_importance'), exist_ok=True)
gbr_imp_df = pd.DataFrame({
    'Feature': importances.index,
    'Importance': importances.values,
    'Percentage': (importances.values * 100).round(2)
})
gbr_imp_df.to_csv(os.path.join('results', 'feature_importance', 'gradient_boosting_importance.csv'), index=False)

print("=== Gradient Boosting Feature Importances ===")
print(gbr_imp_df.to_string(index=False))

=== Gradient Boosting Feature Importances ===
            Feature  Importance  Percentage
             milage      0.6697     66.9700
            car_age      0.1952     19.5200
    is_luxury_brand      0.0625      6.2500
   mileage_per_year      0.0459      4.5900
      fuel_Gasoline      0.0100      1.0000
       has_accident      0.0039      0.3900
          is_manual      0.0033      0.3300
      fuel_Electric      0.0027      0.2700
        fuel_Hybrid      0.0018      0.1800
fuel_Plug-In Hybrid      0.0017      0.1700
         fuel_Other      0.0017      0.1700
 fuel_E85 Flex Fuel      0.0009      0.0900
     is_clean_title      0.0007      0.0700


---
## 5. Hyperparameter Tuning (Stage 7: Optimization)

In [12]:
param_dist = {
    'n_estimators': [100, 150, 200],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'subsample': [0.75, 0.85, 1.0],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

search = RandomizedSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=8,
    scoring='r2',
    cv=kf,
    random_state=42,
    n_jobs=-1
)
search.fit(X_train, y_train)

best_params = search.best_params_
print("=== Best Gradient Boosting Hyperparameters ===")
print(f"Best Parameters: {best_params}")
print(f"Best Tuned CV R²: {search.best_score_:.4f}")

=== Best Gradient Boosting Hyperparameters ===
Best Parameters: {'subsample': 0.85, 'n_estimators': 150, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 4, 'learning_rate': 0.05}
Best Tuned CV R²: 0.6685


---
## 6. Tuned Gradient Boosting Evaluation

In [13]:
tuned_gbr = GradientBoostingRegressor(**best_params, random_state=42)

tuned_cv = cross_validate(
    tuned_gbr,
    X_train,
    y_train,
    cv=kf,
    scoring={
        'r2': 'r2',
        'mae': 'neg_mean_absolute_error',
        'rmse': 'neg_root_mean_squared_error'
    },
    n_jobs=-1
)

tuned_cv_r2   = tuned_cv['test_r2'].mean()
tuned_cv_r2_s = tuned_cv['test_r2'].std()
tuned_cv_mae  = -tuned_cv['test_mae'].mean()
tuned_cv_rmse = -tuned_cv['test_rmse'].mean()

tuned_gbr.fit(X_train, y_train)
y_pred_tuned = tuned_gbr.predict(X_test)

tuned_test_r2   = r2_score(y_test, y_pred_tuned)
tuned_test_mae  = mean_absolute_error(y_test, y_pred_tuned)
tuned_test_rmse = root_mean_squared_error(y_test, y_pred_tuned)

y_pred_tuned_dollar = np.expm1(y_pred_tuned)
tuned_test_mae_dollar  = mean_absolute_error(y_test_dollar, y_pred_tuned_dollar)
tuned_test_rmse_dollar = root_mean_squared_error(y_test_dollar, y_pred_tuned_dollar)

print("=== Tuned Gradient Boosting Performance ===")
print(f"Tuned CV R²           : {tuned_cv_r2:.4f} (+/- {tuned_cv_r2_s:.4f})")
print(f"Tuned CV MAE          : {tuned_cv_mae:.4f}")
print(f"Tuned CV RMSE         : {tuned_cv_rmse:.4f}")
print(f"Tuned Test R²         : {tuned_test_r2:.4f}")
print(f"Tuned Test MAE ($)    : ${tuned_test_mae_dollar:,.2f}")
print(f"Tuned Test RMSE ($)   : ${tuned_test_rmse_dollar:,.2f}")

=== Tuned Gradient Boosting Performance ===
Tuned CV R²           : 0.6685 (+/- 0.0180)
Tuned CV MAE          : 0.3745
Tuned CV RMSE         : 0.4856
Tuned Test R²         : 0.6613
Tuned Test MAE ($)    : $18,286.72
Tuned Test RMSE ($)   : $39,806.98


---
## 7. Save Model Evaluation Records

In [14]:
records = [
    {
        'Model': 'Gradient Boosting',
        'Version': 'Baseline',
        'Parameters': 'n_estimators=100, learning_rate=0.1, max_depth=3',
        'CV_R2': base_cv_r2,
        'CV_R2_Std': base_cv_r2_s,
        'CV_MAE': base_cv_mae,
        'CV_RMSE': base_cv_rmse,
        'Test_R2': base_test_r2,
        'Test_MAE': base_test_mae,
        'Test_RMSE': base_test_rmse,
        'Test_MAE_Dollar': base_test_mae_dollar,
        'Test_RMSE_Dollar': base_test_rmse_dollar,
        'Observation': 'Sequential boosting reduces bias; strong baseline across tabular metrics.'
    },
    {
        'Model': 'Gradient Boosting',
        'Version': 'Tuned',
        'Parameters': str(best_params),
        'CV_R2': tuned_cv_r2,
        'CV_R2_Std': tuned_cv_r2_s,
        'CV_MAE': tuned_cv_mae,
        'CV_RMSE': tuned_cv_rmse,
        'Test_R2': tuned_test_r2,
        'Test_MAE': tuned_test_mae,
        'Test_RMSE': tuned_test_rmse,
        'Test_MAE_Dollar': tuned_test_mae_dollar,
        'Test_RMSE_Dollar': tuned_test_rmse_dollar,
        'Observation': 'Tuned with learning rate shrinkage and subsampling for regularized gradient steps.'
    }
]

pd.DataFrame(records).to_csv(os.path.join('results', 'gradient_boosting_results.csv'), index=False)
print("Gradient Boosting evaluation saved to results/gradient_boosting_results.csv")

Gradient Boosting evaluation saved to results/gradient_boosting_results.csv
